# Part 1 — Imports

In [19]:
from pathlib import Path
import torch

from dmpbridge.pdf.page_image_converter import convert_pdf_to_images
from dmpbridge.vision.qwen_structure_detector import detect_structure_from_images
from dmpbridge.vision.qwen_postprocessor import save_qwen_structured_blocks
from dmpbridge.processing.structure_json_builder import save_narrative_json
from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure

# Part 2 — Check GPU

In [20]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

Torch version: 2.6.0+cu124
CUDA available: True
GPU count: 1
0 NVIDIA GeForce RTX 3090


# Part 3 — Paths

In [21]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample3.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

qwen_output_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}.json"
qwen_structured_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}_structured_blocks.json"
qwen_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_qwen.json"

print("Project root:", project_root)
print("PDF path:", pdf_path)
print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

Project root: c:\Users\Nahid\dmpbridge
PDF path: c:\Users\Nahid\dmpbridge\data\raw_pdfs\sample3.pdf
PDF exists: True
Skeleton exists: True


# Part 4 — Convert PDF to page images

In [22]:
image_paths = convert_pdf_to_images(pdf_path, dpi=120)

print("Number of page images:", len(image_paths))

for p in image_paths:
    print(p, p.exists())

[2026-05-08 11:32:40] Converting PDF pages to images: sample3.pdf
[2026-05-08 11:32:40] Saved 3 page images to: C:\Users\Nahid\dmpbridge\data\page_images\sample3
Number of page images: 3
C:\Users\Nahid\dmpbridge\data\page_images\sample3\page_1.png True
C:\Users\Nahid\dmpbridge\data\page_images\sample3\page_2.png True
C:\Users\Nahid\dmpbridge\data\page_images\sample3\page_3.png True


# Part 5 — Run Qwen2-VL structure detection

In [23]:
qwen_results = detect_structure_from_images(
    image_paths=image_paths,
    output_path=qwen_output_path
)

print("Saved Qwen output:", qwen_output_path.exists())
print("Qwen output path:", qwen_output_path)

qwen_results

[2026-05-08 11:32:40] Loading Qwen2-VL model: Qwen/Qwen2-VL-7B-Instruct


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

[2026-05-08 11:33:08] Running Qwen2-VL on page 1: C:\Users\Nahid\dmpbridge\data\page_images\sample3\page_1.png
[2026-05-08 11:38:58] Running Qwen2-VL on page 2: C:\Users\Nahid\dmpbridge\data\page_images\sample3\page_2.png
[2026-05-08 11:44:48] Running Qwen2-VL on page 3: C:\Users\Nahid\dmpbridge\data\page_images\sample3\page_3.png
[2026-05-08 11:45:33] Saved Qwen structure output: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample3.json
Saved Qwen output: True
Qwen output path: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample3.json


[{'document_title': None,
  'sections': [],
  'raw_response': '{\n  "document_title": "CPS 2015",\n  "sections": [\n    {\n      "title": "Roles and responsibilities",\n      "subsections": [\n        {\n          "title": "The Data Management Plan should clearly articulate how the PI and co-PIs plan to manage and disseminate data generated by the project. The plan should outline the rights and obligations of all parties as to their roles and responsibilities in the management and retention of research data, and consider changes that would occur should a PI or co-PI leave the institution or project. Any costs should be explained in the Budget Justification pages."\n        },\n        {\n          "title": "Data will be generated at each of our institutional sites. The PI at each institution will be responsible for following the data management guidelines described below, and for saving the data to the RIC servers. This process of transferring all data to a central location will be fac

# Part 6 — Print Qwen output clearly

In [24]:
for page in qwen_results:
    print("\nPAGE:", page.get("page"))

    if "error" in page:
        print("ERROR:", page["error"])
        print(page.get("raw_response", "")[:1000])

    for item in page.get("items", []):
        print(item.get("label"), "→", item.get("text"))


PAGE: 1
ERROR: Could not parse Qwen output as JSON
{
  "document_title": "CPS 2015",
  "sections": [
    {
      "title": "Roles and responsibilities",
      "subsections": [
        {
          "title": "The Data Management Plan should clearly articulate how the PI and co-PIs plan to manage and disseminate data generated by the project. The plan should outline the rights and obligations of all parties as to their roles and responsibilities in the management and retention of research data, and consider changes that would occur should a PI or co-PI leave the institution or project. Any costs should be explained in the Budget Justification pages."
        },
        {
          "title": "Data will be generated at each of our institutional sites. The PI at each institution will be responsible for following the data management guidelines described below, and for saving the data to the RIC servers. This process of transferring all data to a central location will be facilitated by our resea

# Part 7 — Convert Qwen output to structured blocks

In [25]:
blocks = save_pdfplumber_outputs(pdf_path)
structured_blocks = detect_structure(blocks)

print("Rule-based structural labels:")

for block in structured_blocks:
    if block["label"] in ["document_title", "section", "subsection", "question"]:
        print(block["label"], "→", block["text"])

[2026-05-08 11:45:33] Extracting line-level text with pdfplumber: sample3.pdf
[2026-05-08 11:45:34] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample3.json
[2026-05-08 11:45:34] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample3.txt
Rule-based structural labels:
document_title → CPS 2015


# Part 8 — Convert Qwen output to structured blocks

In [26]:
qwen_structured_blocks = save_qwen_structured_blocks(
    qwen_output_path=qwen_output_path,
    output_path=qwen_structured_path,
    source_pdf=pdf_path.name
)

print("Saved Qwen structured blocks:", qwen_structured_path.exists())
print("Number of Qwen structured blocks:", len(qwen_structured_blocks))

qwen_structured_blocks[:10]

[2026-05-08 11:45:34] Saved Qwen structured blocks: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample3_structured_blocks.json
Saved Qwen structured blocks: True
Number of Qwen structured blocks: 1


[{'source_pdf': 'sample3.pdf',
  'page': 3,
  'line_order': 1,
  'text': 'Additional possible data management requirements',
  'label': 'section',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'}]

# Part 9 — Build narrative JSON from Qwen blocks

In [27]:
qwen_json = save_narrative_json(
    structured_blocks=qwen_structured_blocks,
    output_path=qwen_json_path,
    skeleton_path=skeleton_path
)

print("Saved Qwen narrative JSON:", qwen_json_path.exists())
print("Qwen JSON path:", qwen_json_path)

[2026-05-08 11:45:34] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample3_qwen.json
Saved Qwen narrative JSON: True
Qwen JSON path: c:\Users\Nahid\dmpbridge\data\structure_json\sample3_qwen.json


# Part 10 — Inspect Qwen narrative JSON

In [28]:
sections = qwen_json["narrative"]["template"]["section"]

print("Number of sections:", len(sections))

for section in sections:
    print(section["order"], section["title"], "| questions:", len(section["question"]))

Number of sections: 1
1 Additional possible data management requirements | questions: 0


In [29]:
sections[0] if sections else "No sections created"

{'id': 'section_1',
 'title': 'Additional possible data management requirements',
 'description': None,
 'order': 1,
 'question': []}